In [ ]:
import numpy as np
import cv2
import glob

# Parametros del tablero (Esquinas internas: donde se cruzan los cuadros negros y blancos)
columnas_internas = 10
filas_internas = 8
cuadro_size_mm = 15

# Criterios de terminacion para la optimizacion (CORREGIDO)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# Preparar los puntos de objeto (0,0,0), (15,0,0), (30,0,0),...
objp = np.zeros((filas_internas * columnas_internas, 3), np.float32)
# Generamos la malla adaptada correctamente a las dimensiones
objp[:, :2] = np.mgrid[0:columnas_internas, 0:filas_internas].T.reshape(-1, 2)
objp *= cuadro_size_mm
import cv2 
import numpy as np

# 1. Cargar datos de calibración
datos = np.load('calibracion_calibracion.npz')
mtx = datos['matriz']
dist = datos['distorsion']

# 2. Inicializar cámara
cap = cv2.VideoCapture(0)

ret, frame = cap.read()
if not ret:
    print("Error al acceder a la cámara")
    exit()

# Dimensiones originales del video
h, w = frame.shape[:2]

# 3. Calcular matrices de mapeo (Se hace una sola vez fuera del bucle para ahorrar CPU)
newcameramtx, roi = cv2.getOptimalNewCameraMatrix(mtx, dist, (w, h), 0, (w, h))
mapx, mapy = cv2.initUndistortRectifyMap(mtx, dist, None, newcameramtx, (w, h), 5)

# Desempaquetar el ROI con nombres claros para no sobreescribir 'w' y 'h'
x_roi, y_roi, w_roi, h_roi = roi

print("Iniciando flujo de video corregido, presiona Enter para salir")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Se perdió la conexión con la cámara")
        break

    # Aplicar la corrección de distorsión
    frame_corregido = cv2.remap(frame, mapx, mapy, cv2.INTER_LINEAR)

    # Recortar la imagen si el ROI es válido (elimina los bordes negros)
    if w_roi > 0 and h_roi > 0:
        frame_corregido = frame_corregido[y_roi:y_roi+h_roi, x_roi:x_roi+w_roi]

    # Mostrar las ventanas
    cv2.imshow("Original con distorsion de barril", frame)
    cv2.imshow("Corregida", frame_corregido)

    # 13 es la tecla Enter
    if cv2.waitKey(1) & 0xFF == 13:
        break

# --- ESTO ESTABA MAL INDENTADO ---
# Debe ir afuera del bucle While
cap.release()
cv2.destroyAllWindows()
objpoints = []  # Puntos 3D en el mundo real
imgpoints = []  # Puntos 2D en el plano de la imagen

# Cargar las imagenes de calibracion con la ruta correcta
ruta_carpeta = '/home/pacon/Tesis_pacon/Jupyter/Tesis-Proyecto/data/ArUcocalib/'
imagenes = glob.glob(ruta_carpeta + 'frame_calib_*.jpg')

if len(imagenes) == 0:
    print("Error: No se encontraron imágenes con el patrón 'frame_calib_*.jpg'")
    exit()

shape_imagen = None

for frame in imagenes:
    img = cv2.imread(frame)
    if img is None:
        continue
        
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    shape_imagen = gray.shape[::-1]  # Guardamos el tamaño (ancho, alto)

    # Buscar las esquinas del tablero
    ret, corners = cv2.findChessboardCorners(gray, (columnas_internas, filas_internas), None)

    if ret == True:
        objpoints.append(objp)
        # Refinar las coordenadas de las esquinas para mayor precision
        corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
        imgpoints.append(corners2)

        # Opcional: Dibujar y mostrar las esquinas
        cv2.drawChessboardCorners(img, (columnas_internas, filas_internas), corners2, ret)
        cv2.imshow('Calibrando...', img)
        cv2.waitKey(500)

# Al terminar el bucle, cerramos las ventanas de muestra
cv2.destroyAllWindows()

# --- LA CALIBRACIÓN VA AQUÍ (FUERA DEL BUCLE) ---
if len(objpoints) > 0:
    print(f"Procesando calibración con {len(objpoints)} imágenes válidas...")
    ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, shape_imagen, None, None)
    print("\n--- RESULTADOS DE LA CALIBRACIÓN ---")
    print("Matriz Intrínseca (mtx):\n", mtx)
    print("\nCoeficientes de Distorsión (dist):\n", dist)
    print(f"\nError de reproyección total: {ret}")
else:
    print("Error: No se pudieron detectar las esquinas en ninguna de las imágenes.")

'''
Procesando calibración con 33 imágenes válidas...

--- RESULTADOS DE LA CALIBRACIÓN ---
Matriz Intrínseca (mtx):
 [[918.7465565    0.         270.87358098]
 [  0.         920.61622722 223.78382511]
 [  0.           0.           1.        ]]

Coeficientes de Distorsión (dist):
 [[-0.50478208  0.86612519  0.00573664  0.00325532 -2.1152841 ]]

Error de reproyección total: 1.909683550439653
'''

Procesando calibración con 33 imágenes válidas...

--- RESULTADOS DE LA CALIBRACIÓN ---
Matriz Intrínseca (mtx):
 [[918.7465565    0.         270.87358098]
 [  0.         920.61622722 223.78382511]
 [  0.           0.           1.        ]]

Coeficientes de Distorsión (dist):
 [[-0.50478208  0.86612519  0.00573664  0.00325532 -2.1152841 ]]

Error de reproyección total: 1.909683550439653


'\nProcesando calibración con 33 imágenes válidas...\n\n--- RESULTADOS DE LA CALIBRACIÓN ---\nMatriz Intrínseca (mtx):\n [[918.7465565    0.         270.87358098]\n [  0.         920.61622722 223.78382511]\n [  0.           0.           1.        ]]\n\nCoeficientes de Distorsión (dist):\n [[-0.50478208  0.86612519  0.00573664  0.00325532 -2.1152841 ]]\n\nError de reproyección total: 1.909683550439653\n'

In [ ]:
#Guardamos la matriz fr la camara y los coeficientes de distorsion 
np.savez('calibracion_calibracion', matriz=mtx,distorsion=dist)
print("Parametros de caslibracion guardados exitosamente")

Parametros de caslibracion guardados exitosamente


In [ ]:
import cv2 
import numpy as np

# 1. Cargar datos de calibración
datos = np.load('calibracion_calibracion.npz')
mtx = datos['matriz']
dist = datos['distorsion']

# 2. Inicializar cámara
cap = cv2.VideoCapture(0)

ret, frame = cap.read()
if not ret:
    print("Error al acceder a la cámara")
    exit()

# Dimensiones originales del video
h, w = frame.shape[:2]

# 3. Calcular matrices de mapeo (Se hace una sola vez fuera del bucle para ahorrar CPU)
newcameramtx, roi = cv2.getOptimalNewCameraMatrix(mtx, dist, (w, h), 0, (w, h))
mapx, mapy = cv2.initUndistortRectifyMap(mtx, dist, None, newcameramtx, (w, h), 5)

# Desempaquetar el ROI con nombres claros para no sobreescribir 'w' y 'h'
x_roi, y_roi, w_roi, h_roi = roi

print("Iniciando flujo de video corregido, presiona Enter para salir")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Se perdió la conexión con la cámara")
        break

    # Aplicar la corrección de distorsión
    frame_corregido = cv2.remap(frame, mapx, mapy, cv2.INTER_LINEAR)

    # Recortar la imagen si el ROI es válido (elimina los bordes negros)
    if w_roi > 0 and h_roi > 0:
        frame_corregido = frame_corregido[y_roi:y_roi+h_roi, x_roi:x_roi+w_roi]

    # Mostrar las ventanas
    cv2.imshow("Original con distorsion de barril", frame)
    cv2.imshow("Corregida", frame_corregido)

    # 13 es la tecla Enter
    if cv2.waitKey(1) & 0xFF == 13:
        break

# --- ESTO ESTABA MAL INDENTADO ---
# Debe ir afuera del bucle While
cap.release()
cv2.destroyAllWindows()